## 그룹과제-3: ML Pipeline 구축

### 목적
임금(Salary) 회귀 문제에 대해 **Feature Engineering → sklearn Pipeline → OOF·앙상블**까지 한 흐름으로 묶은 팀 파이프라인을 구축한다. 이후 진행한 KML InClass 대회 대비용 연습 과제이다.

### 문제·데이터
- **과제 1·2와 동일:** 직무·인구통계 등 설명변수로 **Salary(RMSE)** 예측
- **입력:** `data/X_train.csv`, `y_train.csv`, `X_test.csv` (수업 제공 데이터, 저장소 미포함)

### 모형
- Boosting(XGBoost, LightGBM, CatBoost), 선형(Ridge, Lasso, ElasticNet), 트리·MLP 등
- OOF 예측, Voting, 가중 평균 등 앙상블 적용

### 평가(당시 수업 기준)
- 제출 RMSE 성능 · Pipeline 활용도

In [ ]:
import pandas as pd
from pandas import Series, DataFrame
import numpy as np
#!pip install category_encoders
import category_encoders as ce

# Visualization
import matplotlib.pylab as plt
from matplotlib import font_manager, rc

font_path = "C:/Windows/Fonts/NGULIM.TTF"
font = font_manager.FontProperties(fname=font_path).get_name()
rc('font', family=font)
import seaborn as sns
%matplotlib inline

# Preprocessing & Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import SelectPercentile

# Hyperparameter Optimization
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Modeling
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.base import ClassifierMixin

# Evaluation
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import log_loss

# Utility
import os
import time
import datetime
import random
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image
import pickle
from tqdm import tqdm
import platform
from itertools import combinations
from scipy.stats.mstats import gmean
import re

# !pip install bayesian-optimization
# from bayes_opt import BayesianOptimization
#!pip install num2words
# from num2words import num2words


#추가한것
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn import set_config
from optuna.distributions import CategoricalDistribution, IntDistribution, FloatDistribution
from optuna.integration import OptunaSearchCV, ShapleyImportanceEvaluator
import re
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate

##코드끝나면 울리는 코드
import winsound as sd
def beepsound():
    fr = 2000    # range : 37 ~ 32767
    du = 1000     # 1000 ms ==1second
    sd.Beep(fr, du) # winsound.Beep(frequency, duration)
# !pip install bayesian-optimization
# from bayes_opt import BayesianOptimization
#!pip install num2words
# from num2words import num2words

# 1. Load data

In [ ]:
X_train = pd.read_csv("../data/X_train.csv", encoding='cp949')
y_train = pd.read_csv('../data/y_train.csv', encoding='cp949').Salary  #ID, Salary

X_test = pd.read_csv("../data/X_test.csv", encoding='euc-kr')

In [ ]:
X_train.info() # "직무태그,근무형태, 어학시험,대학성적" 결측치 존재

In [ ]:
X_test.info()

In [ ]:
sns.distplot(y_train); plt.show() 

---
# 2. Preprocessing

## **1) 결측치 처리**

In [ ]:
# 1.(cat) 직무태그 -> 세부직종 값으로 대체  
X_train['직무태그'] = X_train['직무태그'].fillna('없음')
X_test['직무태그'] = X_test['직무태그'].fillna('없음')

# 2.(cat) 근무형태 -> 최빈값
X_train['근무형태'] = X_train['근무형태'].fillna(X_train['근무형태'].mode()[0])                               
X_test['근무형태'] = X_test['근무형태'].fillna(X_test['근무형태'].mode()[0])      

# 3.(cat) 어학시험 -> '없음' (0이 아닌 cat형으로 바꿔야 함)
X_train['어학시험'] = X_train['어학시험'].fillna('없음')
X_test['어학시험'] = X_test['어학시험'].fillna('없음')

# 4.(num) 대학성적 -> 최빈값
X_train['대학성적'] = X_train['대학성적'].fillna(X_train['대학성적'].mode()[0])
X_test['대학성적'] = X_test['대학성적'].fillna(X_test['대학성적'].mode()[0])

display(X_train.head(), X_test.head())

## **2) Feature Engineering**

> ### **직종, 세부직종**

1. 직종별 세부직종에서 분포가 25% 이하인 값 -> '{해당 직종 이름} + 기타'로 대체
2. 직종의 고유값들을 새로운 컬럼으로 만들고, 해당 컬럼들의 값으로 해당 직종에 해당하는 행의 세부직종 값으로 매핑

=> 두 기능을 하나의 사용자 정의 함수로 구현

In [ ]:
def occupation(df):
    ### 1. 25% 이하인 값을 '{해당 직종 이름} + 기타'로 대체
    for i in df['직종'].unique():
        detail = df[df['직종'] == i]['세부직종'].value_counts()
        selected_detail = detail[detail >= detail.quantile(0.25)].index.tolist()
        df.loc[(df['직종'] == i) & (~df['세부직종'].isin(selected_detail)), '세부직종'] = i + ' 기타'

    ### 2. '직종'의 고유값들을 가져와서 새로운 컬럼 생성
    unique_jobs = df['직종'].unique()
    for job in unique_jobs:
        df[job] = '-'  
    for i, row in df.iterrows():
        df.loc[i, row['직종']] = row['세부직종']   # 해당 직종의 세부직종 값을 해당하는 컬럼에 매핑
    
    ### 3. 세부직종_종류_개수
    df['세부직종_종류_개수'] = df['세부직종'].apply(lambda x: len(x.split('·')))

In [ ]:
occupation(X_train)
occupation(X_test)

> ### **직무태그**

> ### **근무형태**

In [ ]:
X_train['근무형태'] = np.where(X_train.근무형태 == "정규직", '정규직', '비정규직')
X_test['근무형태'] = np.where(X_test.근무형태 == "정규직", '정규직', '비정규직')

> ### **근무지역**

In [ ]:
# 지역 컬럼 + 해외 컬럼 만들기

def clean_location(df):
    unique_locations = df['근무지역'].str.split(',').explode().str.strip().unique()
    countries = ['일본', '인도', '미국', '해외', '캐나다', '말레이시아', '중국', '홍콩',
                    '인도네시아', '대만', '싱가포르', '방글라데시', '프랑스', '필리핀', '러시아']
    df['해외'] = 0

    for location in unique_locations:
        if location.strip() and location not in countries:  # 공백이 아니고 countries 내에 없는 경우
            df[location] = df['근무지역'].apply(lambda x: 1 if location in x else 0)
        elif location.strip():  # 공백이 아니면서 countries 내에 있는 경우
            df.loc[df['근무지역'].str.contains(location, na=False), '해외'] = 1

In [ ]:
clean_location(X_train)
clean_location(X_test)

In [ ]:
X_train.columns[X_train.columns.get_loc('해외'):]

> ### **출신대학**

In [ ]:
def 상위10_대학(df):
    top10_list = ['연세대학교', '성균관대학교', '중앙대학교', '이화여자대학교', '서울과기대학교', '세종대학교', '성신여자대학교', '동덕여자데학교', '인천대학교', '서울여자대학교']
    df['상위10_대학'] = 0  
    for i, 대학 in enumerate(df['출신대학']):
        if 대학 in top10_list:
            df.at[i, '상위10_대학'] = 1  # 해당 행의 '상위10_대학' 값을 1로 설정

In [ ]:
상위10_대학(X_train)
상위10_대학(X_test)

>### **대학전공**

In [ ]:
def major(df):
    df['대학전공'] = df['대학전공'].str.replace(r'[^a-zA-Z0-9가-힣]+|(전공|과|부)$', '', regex=True).str.lower()
    df['대학전공_대분류'] = df['대학전공'].apply(lambda x: x[:2])

In [ ]:
major(X_train)
major(X_test)

> ### **어학시험**

In [ ]:
X_train['어학시험'] = X_train['어학시험'].replace(' ', '없음')
X_test['어학시험'] = X_test['어학시험'].replace(' ', '없음')

> ### **자격증**

In [ ]:
''' 파이프라인에서 인코딩 아무거나 때리기'''

## **3) Feature 생성**

In [ ]:
## 대학성적_근무경력
X_train['대학성적_근무경력'] = X_train['대학성적'] * X_train['근무경력']
X_test['대학성적_근무경력'] = X_test['대학성적'] * X_test['근무경력']

In [ ]:
## 직종_근무형태
X_train['직종_근무형태'] = X_train['직종'] + '_' + X_train['근무형태']
X_test['직종_근무형태'] = X_test['직종'] + '_' + X_test['근무형태']

In [ ]:
## 자격증_근무형태
X_train['자격증_근무형태'] = X_train['자격증'] + '_' + X_train['근무형태']
X_test['자격증_근무형태'] = X_test['자격증'] + '_' + X_test['근무형태']

In [ ]:
## 대학성적_근무지역
X_train['대학성적_근무지역'] = X_train['대학성적'].astype(str) + '_' + X_train['근무지역']
X_test['대학성적_근무지역'] = X_test['대학성적'].astype(str) + '_' + X_test['근무지역']

In [ ]:
## 출신대학_전공
X_train['출신대학_전공'] = X_train['출신대학'] + X_train['대학전공']
X_test['출신대학_전공'] = X_test['출신대학'] + X_test['대학전공']

In [ ]:
# 상위10_대학_성적
X_train['상위10_대학_성적'] = X_train['상위10_대학'] + X_train['대학성적']
X_test['상위10_대학_성적'] = X_test['상위10_대학'] + X_test['대학성적']

In [ ]:
# 세부직종개수_대학성적
X_train['세부직종개수_대학성적'] = X_train['세부직종_종류_개수'] + X_train['대학성적']
X_test['세부직종개수_대학성적'] = X_test['세부직종_종류_개수'] + X_test['대학성적']

---
# 3. 수치형/범주형 피처 분리 & 학습/평가 데이터 분할

In [ ]:
### '근무경력' 컬럼 수치형으로 변환
def convert_to_num(df):
    df['근무경력'] = df['근무경력'].apply(lambda x: int(x.split('년')[0])*12 + int(x.split(' ')[1].replace('개월', '')) 
                                                    if '년' in x else int(x.replace('개월', ''))                )
convert_to_num(X_train)
convert_to_num(X_test)

In [ ]:
display(X_train.head(), X_test.head())

In [ ]:
print(X_train.columns, X_test.columns)

In [ ]:
print(X_train.info())
print(X_test.info())

# <span style="color:red"> 여기 아래 코드 수정함!!!

In [ ]:
### 수치형/범주형 피처 분리 (& 학습/평가 데이터 분할)
numeric_features = ['근무경력','대학성적'] 
categorical_features = ['직종', '세부직종', '직무태그', '근무지역', '출신대학', '대학전공', '어학시험',
                        '문화·예술·신문·방송', '경영·기획·회계·사무', 'IT·게임', '영업·판매·TM', '기술·과학·산업', '재료·화학·섬유·의복',
                        '전문·교육·자격', '건설·기계·전기·전자', '디자인', '통신·모바일', '기타 직종', '세부직종_종류_개수',
                        '해외', '서울', '경기', '부산', '충북', '전국', '전북', '인천', '대전', '기타', '경남', '광주', '충남', '강원',
                        '전남', '울산','경북', '제주', '대구', '상위10_대학', '대학전공_대분류']
binary_features = ['자격증', '근무형태']

X_train = X_train[numeric_features+categorical_features + binary_features]#+binary_features]  # 순서 주의!!!
X_test = X_test[numeric_features+categorical_features + binary_features]#+binary_features]


cat_index = [list(X_train.columns).index(c) for c in categorical_features]
X_train.shape, X_test.shape

In [ ]:
X_train

---
# 4. 파이프라인 구축

<!-- removed broken image attachment -->

In [ ]:
from catboost import CatBoostRegressor 
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
### PCA 차원을 자동으로 결정하는 Custom PCA 전처리기 클래스
class MyPCATransformer(TransformerMixin, BaseEstimator):
    # 전처리기 생성 즉, MyPCATransformer() 호출시 실행
    def __init__(self, sum_explained_variance=0.99):
        self.sum_explained_variance = sum_explained_variance

    # 전처리기의 fit() 호출시 실행
    def fit(self, X, y=None):
        # 먼저, 전체 피처에 대해 PCA 수행(차원 축소 없음)
        max_d = X.shape[1]
        pca = PCA(n_components=max_d).fit(X)
        # 누적된 분산의 설명량이 um_explained_variance 이상 되는 차원을 축소할 차원으로 설정
        cumsum = np.cumsum(pca.explained_variance_ratio_)                 #분산의 설명량을 누적합
        self.num_d = np.argmax(cumsum >= self.sum_explained_variance) + 1 #분산의 설명량이 99%이상 되는 차원의 수
        if self.num_d == 1: self.num_d = max_d
        # 축소할 차원으로 다시 PCA 수행 
        self.pca = PCA(n_components=self.num_d)
        self.pca.fit(X)
        return self
    
    # 전처리기의 transform() 호출시 실행
    def transform(self, X):
        return self.pca.transform(X)

In [ ]:
def remove_outlier(X, q=0.05):  
    df = pd.DataFrame(X)
    return df.apply(lambda x: x.clip(x.quantile(q), x.quantile(1-q)), axis=0).values

In [ ]:
from catboost import CatBoostRegressor 
def remove_outlier(X, q=0.05):  
    df = pd.DataFrame(X)
    return df.apply(lambda x: x.clip(x.quantile(q), x.quantile(1-q)), axis=0).values

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")), 
        ("outlier", FunctionTransformer(remove_outlier, kw_args={'q':0.05})), 
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder", OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = np.nan)),
   
    ]
)
    

binary_transformer = Pipeline(
    steps=[
        ("corpus", FunctionTransformer(lambda x: x.str.replace('·',',').str.split(',').str.join(" "))),
        ("BoW", CountVectorizer()),


        
    ]
)

column_transformer = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("bin1", binary_transformer, binary_features[0])
         
    ]
)

preprocessor = Pipeline(
    steps=[
        ("column", column_transformer),
    ]
)



#model = Pipeline(
#    steps=[
#        ("preprocessor", preprocessor), 
#        ("classifier", CatBoostRegressor(iterations=100000,eval_metric = 'RMSE', depth = 6, early_stopping_rounds = 1000, verbose = 200)),
#        #depth = 6 이 디폴튼데 이거 숫자 바꿔가면서ㅏ 실험해보면 조음
#    ]
#)

set_config(display="diagram")  # To view the text pipeline, change to display='text'.
#model
preprocessor

In [ ]:
scores = cross_val_score(model, X_train, y_train, scoring='neg_mean_squared_error', cv=5)

print("Default LM CV scores: ", np.sqrt(-1*scores))
print("Default LM CV mean = %.2f" % np.sqrt(-1*scores.mean()), "with std = %.2f" % np.sqrt(scores.std()))